# 01 — Eval & Promote

Curate production tickets into a versioned evaluation dataset, run a baseline evaluation, iterate the system prompt (here: make drafts concise), compare the candidate against the baseline, and promote the winner — the agent picks up the new version automatically via the `production` alias, with no code change.

**Prerequisite:** run `00_production_traffic_and_monitoring.ipynb` first — it produces the production-traffic traces this notebook samples from.

## Install dependencies

Same as 00_production_traffic_and_monitoring — needed when running 01 as a standalone job (MLflow 3 GenAI APIs). Skip if the cluster already has them.

In [ ]:
%pip install -U mlflow databricks-sdk openai python-dotenv databricks-agents

In [ ]:
dbutils.library.restartPython()

In [ ]:
import sys, json
from pathlib import Path
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
import mlflow
from src import config
from src.agent import run_agent
from src.scorers import ALL_SCORERS
from mlflow.genai.datasets import create_dataset, get_dataset

mlflow.set_experiment(config.MLFLOW_EXPERIMENT_NAME)
EXPERIMENT_ID = mlflow.get_experiment_by_name(config.MLFLOW_EXPERIMENT_NAME).experiment_id
tickets = [r.asDict(recursive=True) for r in spark.table(config.TICKETS_TABLE).select('ticket_id', 'customer_id', 'order_id', 'subject', 'body').orderBy('ticket_id').collect()]
tickets_by_id = {t['ticket_id']: t for t in tickets}
print(f'{len(tickets)} tickets loaded')

## Step 1 — Select 10 production tickets

Sample 10 tickets from the production traffic `00_production_traffic_and_monitoring` logged, to form the evaluation dataset. Here we take the first 10 by ticket_id; in practice you'd sample by uncertainty, coverage, or recent regressions.

In [ ]:
# Production traffic = traces logged by 00_production_traffic_and_monitoring (eval runs created later carry a run_id, which we filter out below).
all_traces = mlflow.search_traces(experiment_ids=[EXPERIMENT_ID], max_results=500, return_type='list')
prod_traces = [t for t in all_traces if not getattr(t.info, 'run_id', None)]

def trace_to_ticket_id(trace):
    inputs = trace.data.spans[0].inputs if trace.data.spans else {}
    if isinstance(inputs, dict):
        ticket = inputs.get('ticket') or inputs
        if isinstance(ticket, dict):
            return ticket.get('ticket_id')
    return None

eval_ticket_ids = []
for t in prod_traces:
    tid = trace_to_ticket_id(t)
    if tid and tid not in eval_ticket_ids:
        eval_ticket_ids.append(tid)
eval_ticket_ids = sorted(eval_ticket_ids)[:10]
print(f'Selected {len(eval_ticket_ids)} eval tickets:', eval_ticket_ids)

## Step 2 — Curate the eval dataset

Save the 10 tickets as a versioned eval dataset (UC Delta table). Inputs only — the scorers don't require expectations. Re-running candidate prompts against this fixed dataset is what makes runs comparable.

In [ ]:
try:
    eval_dataset = get_dataset(name=config.EVAL_DATASET_TABLE)
    print('Using existing eval dataset', config.EVAL_DATASET_TABLE)
except Exception:
    eval_dataset = create_dataset(name=config.EVAL_DATASET_TABLE, experiment_id=EXPERIMENT_ID)
    print('Created eval dataset', config.EVAL_DATASET_TABLE)

records = []
for tid in eval_ticket_ids:
    t = tickets_by_id[tid]
    records.append({'inputs': {'ticket': {
        'ticket_id': t['ticket_id'], 'customer_id': t['customer_id'],
        'order_id': t['order_id'], 'subject': t['subject'], 'body': t['body'],
    }}})
eval_dataset.merge_records(records)
print(f'Eval dataset populated with {len(records)} records at {config.EVAL_DATASET_TABLE}')

## Iterate on the prompt

Edit `candidate_prompt` below (or use `EXAMPLE_V2_PROMPT` from the next cell), then run the experiment cell to score it against the eval dataset.

To move the scorers:
- `response_length_ok` (verbose flaw): add a hard length cap, e.g. "Keep drafts under 80 words."
- `escalation_correctness` (eager-escalation flaw): replace "when in doubt, escalate" with a tight rubric.
- `professional_tone` (casual-tone flaw): the baseline writes in a slangy, hippie voice — require a professional, business-appropriate tone.
- Watch `helpfulness` — over-tightening conciseness can lower it; that tradeoff is the point.

### Example improved prompt (ready to use)

The cell below defines `EXAMPLE_V2_PROMPT`, a worked example that targets the baseline flaws while protecting helpfulness:

- **Concise drafts** — a hard "under 80 words" cap → raises `response_length_ok`.
- **Selective escalation** — replaces "when in doubt, escalate" with a tight rubric → improves `escalation_correctness`.
- **Professional tone** — replaces the casual/hippie voice with a courteous, business-appropriate tone → raises `professional_tone`.
- **Still helpful** — keeps the "address the specific issue + give a concrete next step" requirement so `helpfulness` doesn't drop under the conciseness pressure.

Use it as-is, edit it, or replace it with your own — then run the experiment cell.

In [ ]:
# Ready-made candidate prompt (run as-is or edit). Targets the baseline flaws
# (verbose drafts, unprofessional tone, eager escalation) while keeping it helpful.
EXAMPLE_V2_PROMPT = """\
You are a customer support specialist for an online retailer. For each ticket, produce a concise, structured triage decision.

Process:
1. Read the ticket and pin down the specific issue(s).
2. Use tools only as needed: lookup_customer for tier/history, check_order_status for any order mentioned, search_kb for relevant policy. Skip tools you don't need.
3. Write a brief, helpful draft reply: address the specific issue, use the facts you retrieved, and give one concrete next step or resolution. Keep it UNDER 80 WORDS, in a professional, courteous, business-appropriate tone \u2014 warm but tight, no slang, no filler, no restating the whole situation back to the customer.
4. Classify into exactly one of: billing, shipping, refund, technical, other.
5. Escalate to a human ONLY when at least one is true:
   - The customer explicitly threatens legal action, chargeback, or churn
   - The issue involves fraud, account security, or safety
   - The issue cannot be resolved with the available tools
   - The customer is enterprise tier AND remains unresolved after lookup
   - The customer has 3 or more prior unresolved tickets
   Otherwise resolve it yourself with the tools and KB. Do NOT escalate routine issues (order status, refund eligibility, password reset, in-SLA shipping delays) just because the customer is frustrated.

Return your final output as JSON with these exact keys:
- "category": one of billing | shipping | refund | technical | other
- "escalate": true or false
- "draft": the response to send (a string, under 80 words)
- "reasoning": one or two sentences justifying the category and escalate decisions
"""
print(f'EXAMPLE_V2_PROMPT ready ({len(EXAMPLE_V2_PROMPT.split())} words)')

In [ ]:
baseline_prompt = (config.PROMPTS_DIR / 'system_prompt_v1.md').read_text()
print('--- Baseline (v1) ---')
print(baseline_prompt)

candidate_prompt = EXAMPLE_V2_PROMPT  # <-- use the example, edit it, or paste your own

# Quick sanity-check on one ticket before running the formal experiment:
sample_ticket = tickets[0]
sample_output = run_agent(sample_ticket, system_prompt=candidate_prompt)
print('\n--- Sample agent output on candidate prompt ---')
print(json.dumps(sample_output, indent=2))

## Step 3 — Baseline vs candidate experiment

Run `mlflow.genai.evaluate` against the eval dataset twice — current production prompt (baseline) and the candidate. Both predict_fns are `@mlflow.trace`-decorated with signature `(ticket)` so rows pair by ticket across runs. Scores with all 5 scorers (`ALL_SCORERS`).

In [ ]:
CANDIDATE_RUN_NAME = 'candidate-v2'  # change as you iterate

@mlflow.trace
def baseline_predict_fn(ticket):
    return run_agent(ticket)  # current production-alias prompt

@mlflow.trace
def candidate_predict_fn(ticket):
    return run_agent(ticket, system_prompt=candidate_prompt)

with mlflow.start_run(run_name='baseline-eval'):
    baseline_results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=baseline_predict_fn, scorers=ALL_SCORERS)

with mlflow.start_run(run_name=CANDIDATE_RUN_NAME):
    mlflow.log_param('candidate_prompt', candidate_prompt[:500])
    candidate_results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=candidate_predict_fn, scorers=ALL_SCORERS)
    print(f'Candidate "{CANDIDATE_RUN_NAME}" metrics:')
    print(candidate_results.metrics if hasattr(candidate_results, 'metrics') else candidate_results)

print('\nCompare baseline-eval vs', CANDIDATE_RUN_NAME, 'in the Runs UI (rows pair by ticket).')

## Step 9 — Promote the winning prompt

If the candidate wins, register it as v2 and move the `production` alias. The agent reads `prompts:/...@production` on every call, so the next invocation uses v2 with no code change.

**Run this cell only when you want to promote.**

In [ ]:
PROMOTE = False  # flip to True only when ready

if PROMOTE:
    new_version = mlflow.genai.register_prompt(
        name=config.PROMPT_NAME,
        template=candidate_prompt,
        commit_message=f'Promoted from {CANDIDATE_RUN_NAME} — beats baseline on the eval scorers',
        tags={'previous_alias_holder': 'v1', 'iteration': CANDIDATE_RUN_NAME},
    )
    print(f'Registered version {new_version.version}')

    mlflow.genai.set_prompt_alias(
        name=config.PROMPT_NAME,
        alias=config.PROMPT_ALIAS,
        version=new_version.version,
    )
    print(f"Alias '{config.PROMPT_ALIAS}' now points to version {new_version.version}")
    print('\nPromoted. The next call to run_agent() uses the new prompt.')
else:
    print('PROMOTE flag is False — no changes made to the prompt registry.')

## Verify the promotion

Re-run the agent on one of the original tickets. Because it loads the prompt by alias, it uses the promoted version automatically.

In [ ]:
if PROMOTE:
    test_ticket = tickets[3]
    live_output = run_agent(test_ticket)  # no system_prompt arg → loads from registry by alias
    print(f'Ticket: {test_ticket["subject"]}')
    print('Agent output with promoted prompt:')
    print(json.dumps(live_output, indent=2))
else:
    print('Promotion was skipped — nothing to verify.')